# Notebook 05 — SHAP Explainability & Final Model Persistence

Objective:
1. Apply SHAP to the final selected model.
2. Produce global and local explanations.
3. Save the final model + preprocessing pipeline.

**Language convention:** SHAP values reflect model behaviour — they do NOT prove causality.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import json
import numpy as np
import pandas as pd

from config import (
    SHAP_DIR, FIGURES_DIR, MODELS_DIR,
    FINAL_MODEL_FILE, PREPROCESSING_FILE, MODEL_METADATA_FILE,
    TARGET_COL,
)
from src.data_processing import load_splits
from src.train import save_model
from src.explain import (
    build_shap_explainer,
    get_transformed_data,
    get_feature_names,
    compute_shap_values,
    plot_global_summary,
    plot_global_bar,
    plot_local_waterfall,
    explain_single,
)

SHAP_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
print('Setup complete.')

In [ ]:
X_train, X_test, y_train, y_test = load_splits()
print(f'Train: {X_train.shape}  Test: {X_test.shape}')

## 1. Load the Final Model

Insert your chosen model from Notebook 04 here.
Example shown for Random Forest — update if your selection differs.

In [ ]:
# ---------------------------------------------------------------------------
# IMPORTANT: Replace `best_rf` below with your actual selected model variable
# from Notebook 04. Re-run the training cell there or reuse the object.
# ---------------------------------------------------------------------------

# If re-running from scratch in this notebook:
from src.feature_engineering import build_rf_pipeline
from src.train import tune_model
from config import RF_PARAM_GRID

FINAL_MODEL_LABEL = 'Random Forest (tuned)'   # update to match actual selection
FINAL_THRESHOLD   = 0.5                        # update from threshold analysis

# Re-tune or load — adjust as needed
final_model, best_params, _ = tune_model(
    build_rf_pipeline(), RF_PARAM_GRID, X_train, y_train
)

print(f'Final model: {FINAL_MODEL_LABEL}')
print(f'Best params: {best_params}')

## 2. Build SHAP Explainer

In [ ]:
# Transform training data through the pipeline's preprocessing steps
X_train_t = get_transformed_data(final_model, X_train)
feature_names = get_feature_names(final_model, X_train)

print(f'Transformed shape: {X_train_t.shape}')
print(f'Feature names: {feature_names[:5]} ...')

explainer = build_shap_explainer(final_model, X_train_t)

## 3. Global Explanations

Computed on a subsample of the test set to keep runtime reasonable.

In [ ]:
X_test_t = get_transformed_data(final_model, X_test)
shap_values, X_shap_subset = compute_shap_values(explainer, X_test_t, max_display_rows=500)

In [ ]:
# Beeswarm summary plot
plot_global_summary(shap_values, X_shap_subset, feature_names)

In [ ]:
# Bar chart of mean |SHAP|
plot_global_bar(shap_values, feature_names)

## 4. Local Explanation — Sample Transactions

Explain individual predictions.
Use appropriate language: 'contributed toward' — NOT 'caused'.

In [ ]:
# Select a fraudulent transaction from the test set
fraud_idx = y_test[y_test == 1].index[0]
X_fraud_sample = X_test.loc[[fraud_idx]]

print('=== Local Explanation: Fraudulent Transaction ===')
explanation_fraud = explain_single(
    final_model, explainer, X_fraud_sample, feature_names, threshold=FINAL_THRESHOLD
)

In [ ]:
# Waterfall plot for the fraudulent transaction
X_fraud_t = get_transformed_data(final_model, X_fraud_sample)
plot_local_waterfall(explainer, X_fraud_t, feature_names, index=0)

In [ ]:
# Select a legitimate transaction
legit_idx = y_test[y_test == 0].index[0]
X_legit_sample = X_test.loc[[legit_idx]]

print('=== Local Explanation: Legitimate Transaction ===')
explanation_legit = explain_single(
    final_model, explainer, X_legit_sample, feature_names, threshold=FINAL_THRESHOLD
)

## 5. Save Final Model and Metadata

In [ ]:
import numpy as np
from src.evaluate import compute_metrics

# Final test set evaluation
y_proba_final = final_model.predict_proba(X_test)[:, 1]
y_pred_final  = (y_proba_final >= FINAL_THRESHOLD).astype(int)
final_metrics = compute_metrics(y_test, y_pred_final, y_proba_final, label=FINAL_MODEL_LABEL)

metadata = {
    'model_label':   FINAL_MODEL_LABEL,
    'threshold':     FINAL_THRESHOLD,
    'best_params':   best_params,
    'final_metrics': final_metrics,
}

save_model(final_model, FINAL_MODEL_FILE, metadata=metadata)
print('Model and metadata saved.')

## 6. SHAP Interpretation Summary

(Complete after running all cells.)

**Global insights:**
- Top features contributing to fraud predictions: (fill in from bar chart)
- V-feature patterns: (fill in from beeswarm)

**Local example — Fraudulent transaction:**
- Fraud probability: ...
- Key contributing features: ...

**Local example — Legitimate transaction:**
- Fraud probability: ...
- Key mitigating features: ...

**Important caveat:** SHAP values explain the model's predictions. They indicate which features the model considered most influential — they do NOT establish that these features cause fraud.